In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [52]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-evento-default") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

BASE_PATH = "/app/data"

In [53]:
df_parcela = spark.read.parquet(
    f"{BASE_PATH}/bronze/parcela"
)

df_contrato = spark.read.parquet(
    f"{BASE_PATH}/bronze/contrato"
)

In [54]:
df_default_candidato = df_parcela \
    .filter(col("status_parcela") == "atrasada") \
    .withColumn(
        "dias_atraso",
        datediff(current_date(), col("data_vencimento"))
    ) \
    .filter(
        (col("dias_atraso") > 90) &
        (col("dias_atraso") <= 120)
    )

In [55]:
window_default = Window.partitionBy("contrato_id") \
    .orderBy(desc("dias_atraso"))

In [56]:
df_default_candidato = df_default_candidato \
    .withColumn("rn", row_number().over(window_default)) \
    .filter(col("rn") == 1)

In [57]:
df_evento_default = df_default_candidato.join(
    df_contrato.select(
        "contrato_id",
        "cliente_id",
        "valor_contratado"
    ),
    "contrato_id"
)

In [58]:
df_evento_default = df_evento_default.select(
    monotonically_increasing_id().alias("default_id"),
    col("contrato_id"),
    col("cliente_id"),
    current_date().alias("data_default"),
    col("dias_atraso"),
    round(
        col("valor_contratado") * (rand() * 0.8 + 0.1),
        2
    ).alias("saldo_em_aberto")
)

In [65]:
df_evento_default.write \
    .mode("overwrite") \
    .parquet(f"{BASE_PATH}/bronze/evento_default")
